# Orbit Wars -- Self-Play PPO (pure PyTorch port of the native trainer)

A **single, self-contained** notebook that reproduces the Orbit Wars RL training setup of
`docs/set-ups/1.md` in **pure PyTorch** -- no native C++/LibTorch build, no
`kaggle_environments`. It runs on a simple cloud GPU (Colab / Kaggle Notebooks). It is a
faithful port of the native trainer under `native/` (`gpu_env.cpp`, `policy_net.cpp`,
`distribution.hpp`, `rollout.cpp`, `grpo_trainer.cpp`).

## What it trains
A continuous, per-planet actor-critic on a fully-batched, fully-on-GPU Orbit Wars simulator
(Structure-of-Arrays over `B` envs). The policy is a **squashed-diagonal-Gaussian** over an
`(B, E, 3K)` action: per planet, `K` fleets, each `(dx, dy, phi)`. Aiming is linear --
`theta = atan2(2*dy-1, 2*dx-1)` -- and `phi` is the launch fraction (a fleet commits when
`phi >= act_threshold`, sending `floor(phi * ships)`). Optimizer: **PPO + GAE**.

## Reward (`docs/set-ups/1.md`, REPLACES any production shaping)
1. win `+1000`, loss `-300`, draw `0`.
2. the win reward decays by episode length: `win_value = 1000 * 0.995^steps`. loss is a flat `-300`.
3. capture a planet `+50`, lose a planet `-50` (ego ownership flips, per step).
4. the first **50** committed legal ego launches get `+1` each per game.
5. any **ego** fleet that hits a planet (any owner) gets `5 + ships_on_fleet`, accumulated and
   capped at `300` per game.

The PPO per-step reward **and** the outcome are rescaled by `/100` before GAE (value-target
stability); advantages are re-normalized so the policy objective is unchanged. Logged returns
are in **real (unscaled)** units.

## Curriculum (iteration-driven, 4 stages)
| stage | opponent | iters |
|-------|----------|-------|
| 1 | stationary / noop | 100 |
| 2 | random | 100 |
| 3 | starter (nearest static non-owned) | 200 |
| 4 | self-play + starter mix (`selfplay_prob 0.5`) | 1000 |

The self-play opponent is a **frozen snapshot** of the current policy, acting on the player-1
observation (`encode(ego=1)`), refreshed every 100 iters.

## Simplifications vs. the native engine
- **Comets are OMITTED** (`COMET_SLOTS = 0`; no comet schedule). The native env reserves the
  last 4 planet slots for a periodically-spawned comet group; a comet-free env still trains
  faithfully and removes a large chunk of the step logic. Everything else (orbit, swept
  fleet/planet collision, OOB/sun removal, two-player combat) is ported 1:1.
- **World generator** is a pure-Python/torch mirror of the official `generate_planets`
  (4-fold symmetric groups, `MIN/MAX_PLANET_GROUPS`, polar static groups, ships/production
  ranges) + home assignment. Faithful to the competition distribution; comet schedule dropped.

> Do NOT run heavy training on import. The final smoke cell is guarded by `RUN_SMOKE=False`.


## 1. Imports
Only `torch`, `numpy`, `matplotlib` -- all preinstalled on Colab / Kaggle.

In [ ]:
import math
import random
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.set_float32_matmul_precision("high")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32  # integer-in-float32 convention (mirrors the native env)
print("device:", DEVICE, "| torch", torch.__version__)

## 2. Config (editable hyperparameters)

Mirrors `scripts/run_setup1.cmd`. Set `QUICK_SMOKE = True` for a fast Colab/Kaggle sanity run
(smaller model, fewer iters, smaller batch). Set `False` for the full spec.

Native run uses `B=128` (`num_groups 16 * group_size 8`), width `512`, `20` res blocks.
We keep `B=128` as the default but note you can drop it to fit VRAM. PPO has no groups (the
critic is the baseline), so `B` is just the env count.

In [ ]:
# ---- master switch: shrink everything for a quick sanity run -----------------
QUICK_SMOKE = True   # <<< set False for the full docs/set-ups/1.md spec

# ---- model size (full spec: width 512, 20 residual blocks) -------------------
HIDDEN          = 256 if QUICK_SMOKE else 512   # d: trunk width  ("model dim")
N_RES_BLOCKS    = 8   if QUICK_SMOKE else 20    # "hidden = 10" -> reinterpreted as 20 res blocks
D_G             = 32                            # board-globals embedding dim
USE_GLU         = True
STD_STATE_DEP   = True                          # state-dependent logstd head

# ---- env / batch ------------------------------------------------------------
B               = 32  if QUICK_SMOKE else 128   # number of parallel envs (reduce to fit VRAM)
EPISODE_STEPS   = 120 if QUICK_SMOKE else 500
K_FLEETS        = 5                             # K fleets/planet -> 3K action params
PLANET_CAP      = 48                            # E: planet slots/env (no comet slots: comets omitted)
FLEET_CAP       = 512 if QUICK_SMOKE else 1024  # max simultaneous in-flight fleets/env
ACT_THRESHOLD   = 0.05                          # tau_act: commit when phi >= this
SHIP_SPEED      = 6.0
COMET_SLOTS     = 0                             # comets OMITTED (see intro)

# ---- PPO + GAE --------------------------------------------------------------
LR              = 1e-4
GAMMA           = 0.99
GAE_LAMBDA      = 0.95
CLIP            = 0.2
VF_COEF         = 0.5
ENT_COEF        = 0.005
MINIBATCHES     = 16  if QUICK_SMOKE else 64
UPDATE_EPOCHS   = 1
MAX_GRAD_NORM   = 0.5
ADAM_EPS        = 1e-5

# ---- logstd cap anneal: 0 -> -1.2 over the first 100 iters, then -1.0 --------
LOGSTD_MIN      = -2.0
LOGSTD_MAX      = 0.0      # phase-1 start
LOGSTD_MAX_END  = -1.2     # phase-1 forced-decay target
LOGSTD_MAX_POST = -1.0     # phase-2 cap (head takes over)
SIGMA_DECAY_ITERS = 100    # phase-1 length in iterations

# ---- calm init --------------------------------------------------------------
INIT_MU_SCALE   = 0.02     # small final mu-layer weights
INIT_PHI_BIAS   = -4.0     # negative phi bias (few commits at init)

# ---- curriculum (iters per stage) -------------------------------------------
if QUICK_SMOKE:
    STAGE1_ITERS, STAGE2_ITERS, STAGE3_ITERS, TOTAL_ITERS = 10, 10, 20, 60
else:
    STAGE1_ITERS, STAGE2_ITERS, STAGE3_ITERS, TOTAL_ITERS = 100, 100, 200, 1400
SELFPLAY_PROB     = 0.5
SELFPLAY_REFRESH  = 100 if not QUICK_SMOKE else 20   # refresh frozen snapshot every N iters

# ---- reward (docs/set-ups/1.md) ---------------------------------------------
WIN_BONUS         = 1000.0
LOSS_PENALTY      = 300.0
WIN_DECAY         = 0.995    # win value = WIN_BONUS * WIN_DECAY^len
CAPTURE_REWARD    = 50.0     # +/- per planet gained/lost
DISPATCH_REWARD   = 1.0      # +1 each for the first DISPATCH_COUNT legal launches
DISPATCH_COUNT    = 50
FLEET_HIT_BASE    = 5.0      # per ego fleet hit: base + ship_weight*ships
FLEET_HIT_SHIPW   = 1.0
FLEET_HIT_CAP     = 300.0    # per-game cap
PPO_REWARD_SCALE  = 100.0    # divide per-step reward AND outcome by this before GAE

# ---- world pool -------------------------------------------------------------
N_WORLDS        = 256 if QUICK_SMOKE else 2048
SEED            = 0

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("QUICK_SMOKE =", QUICK_SMOKE, "| B =", B, "| HIDDEN =", HIDDEN,
      "| N_RES_BLOCKS =", N_RES_BLOCKS, "| TOTAL_ITERS =", TOTAL_ITERS)

## 3. Feature layout (mirrors `core/encode.hpp` + `core/state.hpp`)

`F = 11 body + 21 threat = 32` features/planet; `G = 10` board globals (a separate input).
Board constants are the competition's. The threat features keep the `N_SOON=2` soonest +
`N_BIG=5` largest inbound fleets per planet, 3 feats each.

In [ ]:
# board constants (core/state.hpp)
BOARD_SIZE = 100.0
CENTER = BOARD_SIZE / 2.0
SUN_RADIUS = 10.0
ROTATION_RADIUS_LIMIT = 50.0
COMET_RADIUS = 1.0
COMET_PRODUCTION = 1
PI = math.pi

# feature layout (core/encode.hpp)
N_SOON = 2
N_BIG = 5
N_THREAT_FLEETS = N_SOON + N_BIG            # 7 slots * 3 feats = 21
N_BODY_FEATURES = 11
N_ENTITY_FEATURES = N_BODY_FEATURES + 3 * N_THREAT_FLEETS   # 32
N_GLOBAL_FEATURES = 10
F_DIM = N_ENTITY_FEATURES
G_DIM = N_GLOBAL_FEATURES

SHIP_LOG_DENOM = math.log(1000.0)
DIAG_HALF = math.sqrt(BOARD_SIZE * BOARD_SIZE + BOARD_SIZE * BOARD_SIZE) / 2.0
THREAT_MAX_SPEED = 6.0      # == ship speed; fleet speed cap for the ETA model
THREAT_ETA_SCALE = 100.0
BIG = 1e18

def ship_log_t(x):
    return torch.log1p(x.clamp_min(0.0)) / SHIP_LOG_DENOM

def fleet_speed_t(ships, vmax=SHIP_SPEED):
    # 1 + (vmax-1)*(log(ships)/log(1000))^1.5, capped, ships>=1  (core/encode.hpp)
    n = ships.clamp_min(1.0)
    v = 1.0 + (vmax - 1.0) * torch.pow(torch.log(n) / math.log(1000.0), 1.5)
    return v.clamp_max(vmax)

print("F =", F_DIM, "| G =", G_DIM, "| action params/planet = 3K =", 3 * K_FLEETS)

## 4. Squashed diagonal Gaussian (mirrors `model/distribution.hpp`)

Per component `a = sigmoid(z)`, `z ~ N(mu, sigma^2)`, `sigma = exp(logstd)`. Joint log-prob and
entropy are **sums over all `E` planet slots and all `3K` params** (unmasked: legality is
learned from the dispatch penalty / spec reward). `log_prob` includes the sigmoid
change-of-variables (log-det-Jacobian) correction.

In [ ]:
_LOG2PI = 1.8378770664093453      # log(2*pi)
_HALF_LOG2PIE = 1.4189385332046727  # 0.5*log(2*pi*e)

class SquashedGaussian:
    '''a = sigmoid(z), z ~ N(mean, exp(logstd)^2). mean/logstd: (B,E,3K).'''
    def __init__(self, mean, logstd):
        self.mean = mean
        self.logstd = logstd
        self.std = torch.exp(logstd)

    def sample(self):
        return torch.sigmoid(self.mean + self.std * torch.randn_like(self.mean))

    def greedy(self):
        return torch.sigmoid(self.mean)

    def log_prob(self, a):
        ac = a.clamp(1e-6, 1.0 - 1e-6)
        z = torch.log(ac) - torch.log1p(-ac)        # logit(a) recovers latent z
        mu = self.mean.clamp(-15.0, 15.0)           # bound (z-mu)^2 if a head emits an extreme mean
        logN = -0.5 * (z - mu).pow(2) / (self.std * self.std) - self.logstd - 0.5 * _LOG2PI
        logjac = torch.log(ac) + torch.log1p(-ac)   # log|da/dz| = log(a(1-a))
        term = logN - logjac                        # (B,E,3K)
        return term.sum(2).sum(1)                    # (B,)

    def entropy(self):
        e = self.logstd + _HALF_LOG2PIE
        return e.sum(2).sum(1)

## 5. World generator (pure Python mirror of official `generate_planets`)

Mirrors `REFERENCE_orbit_wars.py::generate_planets` + `native_worldgen.generate_world`:
4-fold symmetric planet groups (each `[id, owner, x, y, radius, ships, production]`), a
guaranteed `MIN_STATIC_GROUPS=3` static groups via polar sampling, `MIN/MAX_PLANET_GROUPS`
total, then home assignment (`base` group: planet 0 -> player 0 with 10 ships, planet 3 ->
player 1 with 10 ships). `angular_velocity ~ U(0.025, 0.05)`.

**Simplification:** no comet schedule (comets omitted). Worlds are packed straight into the
batched env tensors (no `.owp` files).

In [ ]:
MIN_PLANET_GROUPS = 5
MAX_PLANET_GROUPS = 10
MIN_STATIC_GROUPS = 3
PLANET_CLEARANCE = 7

def _dist(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])

def generate_planets(rng):
    '''Faithful mirror of REFERENCE_orbit_wars.py::generate_planets.
    Returns rows [id, owner, x, y, radius, ships, production].'''
    planets = []
    num_q1 = rng.randint(MIN_PLANET_GROUPS, MAX_PLANET_GROUPS)
    idc = 0
    # Phase 1: guaranteed static groups (polar sampling).
    static_groups = 0
    for _ in range(5000):
        if static_groups >= MIN_STATIC_GROUPS:
            break
        prod = rng.randint(1, 5)
        r = 1 + math.log(prod)
        angle = rng.uniform(0, math.pi / 2)
        min_orbital = ROTATION_RADIUS_LIMIT - r
        max_orbital = (BOARD_SIZE - CENTER - r) / max(math.cos(angle), math.sin(angle))
        if min_orbital > max_orbital:
            continue
        orbital_r = rng.uniform(min_orbital, max_orbital)
        x = CENTER + orbital_r * math.cos(angle)
        y = CENTER + orbital_r * math.sin(angle)
        if x + r > BOARD_SIZE or x - r < 0 or y + r > BOARD_SIZE or y - r < 0:
            continue
        if (BOARD_SIZE - x) - r < 0 or (BOARD_SIZE - y) - r < 0:
            continue
        if (x - CENTER) < r + 5 or (y - CENTER) < r + 5:
            continue
        ships = min(rng.randint(5, 99), rng.randint(5, 99))
        # NOTE: the reference stores rows as [id, owner, y, x, r, ...] (x/y swapped naming),
        # which is just a relabel of the symmetric copies; we keep it identical.
        tps = [
            [idc, -1, y, x, r, ships, prod],
            [idc + 1, -1, BOARD_SIZE - x, y, r, ships, prod],
            [idc + 2, -1, x, BOARD_SIZE - y, r, ships, prod],
            [idc + 3, -1, BOARD_SIZE - y, BOARD_SIZE - x, r, ships, prod],
        ]
        valid = True
        for tp in tps:
            for p in planets:
                if _dist((p[2], p[3]), (tp[2], tp[3])) < p[4] + tp[4] + PLANET_CLEARANCE:
                    valid = False; break
            if not valid:
                break
        if valid:
            planets.extend(tps); idc += 4; static_groups += 1
    # Phase 2: fill remaining groups (normal random loop).
    attempts = 0
    max_attempts = 5000
    has_orbiting = False
    while len(planets) < num_q1 * 4 or (not has_orbiting and attempts < max_attempts):
        attempts += 1
        if attempts >= max_attempts:
            break
        prod = rng.randint(1, 5)
        r = 1 + math.log(prod)
        x = rng.uniform(CENTER + 15, BOARD_SIZE - r - 5)
        y = rng.uniform(CENTER + 15, BOARD_SIZE - r - 5)
        orbital_radius = _dist((x, y), (CENTER, CENTER))
        if orbital_radius < SUN_RADIUS + r + 10:
            continue
        if orbital_radius + r >= ROTATION_RADIUS_LIMIT:
            if x + r > BOARD_SIZE or x - r < 0 or y + r > BOARD_SIZE or y - r < 0:
                continue
        valid = True
        ships = rng.randint(5, 30)
        tps = [
            [idc, -1, y, x, r, ships, prod],
            [idc + 1, -1, BOARD_SIZE - x, y, r, ships, prod],
            [idc + 2, -1, x, BOARD_SIZE - y, r, ships, prod],
            [idc + 3, -1, BOARD_SIZE - y, BOARD_SIZE - x, r, ships, prod],
        ]
        for tp in tps:
            tp_orb = _dist((tp[2], tp[3]), (CENTER, CENTER))
            tp_rot = tp_orb + tp[4] < ROTATION_RADIUS_LIMIT
            for p in planets:
                p_orb = _dist((p[2], p[3]), (CENTER, CENTER))
                p_rot = p_orb + p[4] < ROTATION_RADIUS_LIMIT
                if _dist((p[2], p[3]), (tp[2], tp[3])) < p[4] + tp[4] + PLANET_CLEARANCE:
                    valid = False; break
                if tp_rot != p_rot:
                    if abs(tp_orb - p_orb) < tp[4] + p[4] + PLANET_CLEARANCE:
                        valid = False; break
            if not valid:
                break
        if valid:
            if orbital_radius + r < ROTATION_RADIUS_LIMIT:
                has_orbiting = True
            planets.extend(tps); idc += 4
    return planets

def generate_world(seed):
    '''Mirror of native_worldgen.generate_world (comets dropped).'''
    rng = random.Random(seed)
    angular_velocity = rng.uniform(0.025, 0.05)
    planets = generate_planets(rng)
    num_groups = len(planets) // 4
    if num_groups > 0:
        base = rng.randint(0, num_groups - 1) * 4
        planets[base][1] = 0;      planets[base][5] = 10       # player 0 home
        planets[base + 3][1] = 1;  planets[base + 3][5] = 10   # player 1 home
    return {"planets": planets, "angular_velocity": angular_velocity}

def make_world_pool(n, base_seed=0):
    return [generate_world(base_seed + i) for i in range(n)]

# quick sanity: one world's shape
_w = generate_world(0)
print("world 0:", len(_w["planets"]), "planets | ang_vel = %.4f" % _w["angular_velocity"])

## 6. Batched env (mirrors `rl/gpu_env.cpp`)

Structure-of-Arrays over `B` envs, leading dim `B`. Integer quantities (ships, production,
owner, step) live in float32 (exact below `2^24`, cheaper on GPU). Planets in fixed slots
`[0, PLANET_CAP)`; fleets in a fixed pool `[0, FLEET_CAP)` with an alive mask. **No comet
slots** (comets omitted).

`reset` packs a list of world dicts into the device tensors and zeroes runtime state.

In [ ]:
class GpuEnv:
    def __init__(self, planet_cap, fleet_cap, episode_steps, ship_speed, device):
        self.Ec = planet_cap
        self.Fc = fleet_cap
        self.T = episode_steps
        self.vmax = ship_speed
        self.dev = device
        self.B = 0

    def reset(self, worlds):
        B = len(worlds)
        self.B = B
        Ec, Fc = self.Ec, self.Fc
        dev = self.dev
        z = lambda *s: torch.zeros(s, dtype=DTYPE, device=dev)
        # planets (B, Ec)
        self.p_alive = z(B, Ec)
        self.p_owner = torch.full((B, Ec), -1.0, dtype=DTYPE, device=dev)
        self.p_x = z(B, Ec); self.p_y = z(B, Ec); self.p_radius = z(B, Ec)
        self.p_ships = z(B, Ec); self.p_prod = z(B, Ec); self.p_is_comet = z(B, Ec)
        self.p_init_x = z(B, Ec); self.p_init_y = z(B, Ec); self.p_rotates = z(B, Ec)
        # fill from worlds (CPU numpy then upload once)
        pa = np.zeros((B, Ec), np.float32); po = np.full((B, Ec), -1.0, np.float32)
        px = np.zeros((B, Ec), np.float32); py = np.zeros((B, Ec), np.float32)
        pr = np.zeros((B, Ec), np.float32); ps = np.zeros((B, Ec), np.float32)
        pp = np.zeros((B, Ec), np.float32); pix = np.zeros((B, Ec), np.float32)
        piy = np.zeros((B, Ec), np.float32); prot = np.zeros((B, Ec), np.float32)
        angv = np.zeros((B,), np.float32)
        for b, w in enumerate(worlds):
            pls = w["planets"][:Ec]
            for i, pl in enumerate(pls):
                pid, owner, x, y, rad, sh, prod = pl
                pa[b, i] = 1.0; po[b, i] = float(owner)
                px[b, i] = x; py[b, i] = y; pr[b, i] = rad
                ps[b, i] = sh; pp[b, i] = prod
                pix[b, i] = x; piy[b, i] = y
                rr = math.hypot(x - CENTER, y - CENTER)
                prot[b, i] = 1.0 if (rr + rad < ROTATION_RADIUS_LIMIT) else 0.0
            angv[b] = w["angular_velocity"]
        t = lambda a: torch.from_numpy(a).to(dev)
        self.p_alive = t(pa); self.p_owner = t(po); self.p_x = t(px); self.p_y = t(py)
        self.p_radius = t(pr); self.p_ships = t(ps); self.p_prod = t(pp)
        self.p_init_x = t(pix); self.p_init_y = t(piy); self.p_rotates = t(prot)
        self.p_is_comet = z(B, Ec)
        # fleets (B, Fc)
        self.f_alive = z(B, Fc); self.f_owner = z(B, Fc); self.f_x = z(B, Fc)
        self.f_y = z(B, Fc); self.f_angle = z(B, Fc); self.f_ships = z(B, Fc)
        self.f_seq = z(B, Fc)
        # per-env
        self.ang_vel = t(angv)
        self.step_ct = torch.zeros(B, dtype=DTYPE, device=dev)
        self.done = torch.zeros(B, dtype=DTYPE, device=dev)

## 7. Env -- `fleet_target_batch` + `encode`

`fleet_target_batch`: closed-form ray-vs-disk + sun occlusion for `M` virtual fleets/env vs
the current planets (mirrors `gpu_env.cpp::fleet_target_batch`). Used by `encode` (threat
features, `M=Fc`) and by the valid-launch check.

`encode(ego)`: the batched mirror of `encode_obs`. Planets stay in fixed slots (the per-planet
actor is permutation-equivariant). Returns entities `(B,Ec,F)`, entity_mask, action_mask,
globals `(B,Ec...)`.

In [ ]:
def fleet_target_batch(env, fx, fy, fang, fships, vmax):
    '''fx,fy,fang,fships: (B,M). Returns tgt (B,M) long (planet slot or -1), eta (B,M).'''
    dx = torch.cos(fang).unsqueeze(2)           # (B,M,1)
    dy = torch.sin(fang).unsqueeze(2)
    pxr = env.p_x.unsqueeze(1)                  # (B,1,Ec)
    pyr = env.p_y.unsqueeze(1)
    rr = (env.p_radius * env.p_radius).unsqueeze(1)
    alive = (env.p_alive > 0.5).unsqueeze(1)
    ox = fx.unsqueeze(2) - pxr                  # (B,M,Ec)
    oy = fy.unsqueeze(2) - pyr
    tca = -(ox * dx + oy * dy)
    perp2 = ox * ox + oy * oy - tca * tca
    hit = (tca >= 0.0) & (perp2 <= rr) & alive
    t_int = (tca - torch.sqrt((rr - perp2).clamp_min(0.0))).clamp_min(0.0)
    tvals = torch.where(hit, t_int, torch.full_like(t_int, BIG))
    best_t, best_e = tvals.min(2)               # (B,M)
    any_hit = best_t < BIG
    # sun occlusion
    sx = fx - CENTER; sy = fy - CENTER
    dxx = torch.cos(fang); dyy = torch.sin(fang)
    tcs = -(sx * dxx + sy * dyy)
    sperp2 = sx * sx + sy * sy - tcs * tcs
    sr = SUN_RADIUS * SUN_RADIUS
    t_sun = tcs - torch.sqrt((sr - sperp2).clamp_min(0.0))
    sun_block = (tcs >= 0.0) & (sperp2 <= sr) & (t_sun >= 0.0) & (t_sun < best_t)
    valid = any_hit & (~sun_block)
    tgt = torch.where(valid, best_e, torch.full_like(best_e, -1))
    eta = best_t / fleet_speed_t(fships, vmax)
    eta = torch.where(valid, eta, torch.zeros_like(eta))
    return tgt, eta


def env_encode(env, ego=0):
    B, Ec, Fc = env.B, env.Ec, env.Fc
    enemy = 1 - ego
    na = 2
    dev = env.dev
    alive = env.p_alive > 0.5
    av = env.p_alive
    owner = env.p_owner
    dxc = env.p_x - CENTER; dyc = env.p_y - CENTER
    dist = torch.sqrt(dxc * dxc + dyc * dyc)
    comet = env.p_is_comet > 0.5
    rotating = (~comet) & ((dist + env.p_radius) < ROTATION_RADIUS_LIMIT)
    angv = env.ang_vel.unsqueeze(1)
    vmag = torch.where(rotating, angv.abs() * dist, torch.zeros_like(dist))
    cw = torch.where(rotating,
                     torch.where(angv >= 0.0, torch.ones_like(dist), -torch.ones_like(dist)),
                     torch.zeros_like(dist))
    # ego-relative ownership code: 0 neutral, 1 ego, 2..na enemy
    m = owner - float(ego) + float(na)
    m = m - float(na) * torch.floor(m / float(na))
    own_code = torch.where(owner < 0.0, torch.zeros_like(owner), 1.0 + m)
    b0 = env.p_x / BOARD_SIZE
    b1 = env.p_y / BOARD_SIZE
    b2 = vmag / THREAT_MAX_SPEED
    b3 = cw
    b4 = env.p_radius / 3.0
    b5 = env.p_prod / 5.0
    b6 = ship_log_t(env.p_ships)
    b7 = own_code
    b8 = dist / DIAG_HALF
    b9 = comet.to(DTYPE)
    actable = (owner == float(ego)) & alive & (env.p_ships > 0.0)
    b10 = actable.to(DTYPE)
    body = torch.stack([b0, b1, b2, b3, b4, b5, b6, b7, b8, b9, b10], 2)  # (B,Ec,11)

    # threat: per planet, N_SOON soonest + N_BIG largest inbound fleets
    ftgt, feta = fleet_target_batch(env, env.f_x, env.f_y, env.f_angle, env.f_ships, env.vmax)
    falive = env.f_alive > 0.5
    fsign = torch.where(env.f_owner == float(ego), torch.ones_like(env.f_ships),
                        -torch.ones_like(env.f_ships))
    fslog = ship_log_t(env.f_ships.abs())
    slot = torch.arange(Ec, dtype=DTYPE, device=dev).view(1, Ec, 1)
    tgtf = ftgt.to(DTYPE).unsqueeze(1)                       # (B,1,Fc)
    targeting = (tgtf == slot) & (ftgt >= 0).unsqueeze(1) & falive.unsqueeze(1)  # (B,Ec,Fc)
    W = float(1 << 20)
    eta_mat = torch.where(targeting, feta.unsqueeze(1), torch.full((1,), BIG, device=dev))
    absf = env.f_ships.abs().unsqueeze(1)
    seqf = env.f_seq.unsqueeze(1)
    key_big = torch.where(targeting, absf * W - seqf, torch.full((1,), -BIG, device=dev))
    eta_top_v, eta_top_i = torch.topk(eta_mat, N_SOON, 2, largest=False)
    big_top_v, big_top_i = torch.topk(key_big, N_BIG, 2, largest=True)
    idx = torch.cat([eta_top_i, big_top_i], 2)              # (B,Ec,7)
    valid = torch.cat([eta_top_v < BIG * 0.5, big_top_v > -BIG * 0.5], 2)
    def pick(src):
        return src.unsqueeze(1).expand(B, Ec, Fc).gather(2, idx)
    zt = torch.zeros(B, Ec, N_THREAT_FLEETS, device=dev)
    sgn = torch.where(valid, pick(fsign), zt)
    etf = torch.where(valid, pick(feta) / THREAT_ETA_SCALE, zt)
    slg = torch.where(valid, pick(fslog), zt)
    threat = torch.stack([sgn, etf, slg], 3).reshape(B, Ec, 3 * N_THREAT_FLEETS)  # (B,Ec,21)

    entities = torch.cat([body, threat], 2) * av.unsqueeze(2)   # zero dead slots
    entity_mask = av
    action_mask = b10

    # globals (B,10)
    def psum(mask):
        return (env.p_ships * mask.to(DTYPE)).sum(1)
    mine = (owner == float(ego)) & alive
    en = (owner == float(enemy)) & alive
    neu = (owner < 0.0) & alive
    my_ships = psum(mine); en_ships = psum(en)
    my_pl = mine.to(DTYPE).sum(1); en_pl = en.to(DTYPE).sum(1); neu_pl = neu.to(DTYPE).sum(1)
    npl = av.sum(1); total = npl.clamp_min(1.0)
    my_fleet = (env.f_ships * (env.f_owner == float(ego)).to(DTYPE) * falive.to(DTYPE)).sum(1)
    en_fleet = (env.f_ships * (env.f_owner == float(enemy)).to(DTYPE) * falive.to(DTYPE)).sum(1)
    ME = 40.0
    g0 = env.step_ct / max(1, env.T)
    g1 = env.ang_vel * 10.0
    g2 = ship_log_t(my_ships); g3 = ship_log_t(en_ships)
    g4 = my_pl / total; g5 = en_pl / total; g6 = neu_pl / total
    g7 = ship_log_t(my_fleet); g8 = ship_log_t(en_fleet)
    g9 = torch.minimum(npl, torch.full_like(npl, ME)) / ME
    globals_ = torch.stack([g0, g1, g2, g3, g4, g5, g6, g7, g8, g9], 1)
    return entities, entity_mask, action_mask, globals_

## 8. Env -- `launch_fleets` + scripted opponents

`launch_fleets`: scatter committed launches into free fleet slots (mirrors
`gpu_env.cpp::launch_fleets`; ship deduction happens in `step`). `opponent_action`: noop /
random / starter (mirrors `gpu_env.cpp::opponent_action`).

In [ ]:
def launch_fleets(env, owner, from_slot, angle, ships, commit, seq):
    '''Append committed launches (all (B,L)) into the fleet pool, distinct free slots.'''
    B, Fc = env.B, env.Fc
    dev = env.dev
    L = from_slot.shape[1]
    free = env.f_alive < 0.5
    freef = free.to(DTYPE)
    fr = torch.cumsum(freef, 1) - 1.0
    n_free = freef.sum(1)
    idx = torch.where(free, fr.long(), torch.full_like(fr.long(), Fc))
    rank_to_slot = torch.full((B, Fc + 1), Fc, dtype=torch.long, device=dev)
    slot_src = torch.arange(Fc, dtype=torch.long, device=dev).unsqueeze(0).expand(B, Fc)
    rank_to_slot.scatter_(1, idx, slot_src)
    rank_to_slot = rank_to_slot[:, :Fc]
    commitb = commit > 0.5
    crank = (torch.cumsum(commit.to(DTYPE), 1) - 1.0).long()
    place = commitb & (crank < n_free.unsqueeze(1).long())
    gslot = rank_to_slot.gather(1, crank.clamp(0, Fc - 1))
    dump = torch.full_like(gslot, Fc)
    wslot = torch.where(place, gslot, dump)
    def scatter_into(field, vals):
        aug = torch.cat([field, torch.zeros(B, 1, dtype=DTYPE, device=dev)], 1)
        aug.scatter_(1, wslot, vals)
        return aug[:, :Fc]
    fs = from_slot.clamp(0, env.p_x.shape[1] - 1)
    opx = env.p_x.gather(1, fs); opy = env.p_y.gather(1, fs); orad = env.p_radius.gather(1, fs)
    sx = opx + torch.cos(angle) * (orad + 0.1)
    sy = opy + torch.sin(angle) * (orad + 0.1)
    onesL = torch.ones(B, L, dtype=DTYPE, device=dev)
    env.f_alive = scatter_into(env.f_alive, onesL)
    env.f_owner = scatter_into(env.f_owner, owner)
    env.f_x = scatter_into(env.f_x, sx)
    env.f_y = scatter_into(env.f_y, sy)
    env.f_angle = scatter_into(env.f_angle, angle)
    env.f_ships = scatter_into(env.f_ships, ships)
    env.f_seq = scatter_into(env.f_seq, seq)


def opponent_action(env, opponent):
    '''Scripted player-1 launches: one per owned planet, half garrison (>=20).
    0=random heading, 1=starter (nearest static non-owned), 2=noop. -> angle,ships,commit (B,Ec).'''
    B, Ec = env.B, env.Ec
    dev = env.dev
    min_ships = 20.0
    angle = torch.zeros(B, Ec, dtype=DTYPE, device=dev)
    ships = torch.zeros(B, Ec, dtype=DTYPE, device=dev)
    commit = torch.zeros(B, Ec, dtype=DTYPE, device=dev)
    if opponent == 2:
        return angle, ships, commit
    half = torch.floor(env.p_ships / 2.0)
    base = (env.p_owner == 1.0) & (env.p_alive > 0.5) & (half >= min_ships)
    if opponent == 0:  # random heading
        angle = torch.rand(B, Ec, dtype=DTYPE, device=dev) * (2.0 * PI)
        ships = torch.where(base, half, torch.zeros_like(half))
        commit = base.to(DTYPE)
        return angle, ships, commit
    # opponent == 1: starter -- fire at nearest static non-owned planet
    distc = torch.sqrt((env.p_x - CENTER) ** 2 + (env.p_y - CENTER) ** 2)
    is_static = (distc + env.p_radius) >= ROTATION_RADIUS_LIMIT
    valid_tgt = is_static & (env.p_alive > 0.5) & (env.p_owner != 1.0)
    sx = env.p_x.unsqueeze(2); sy = env.p_y.unsqueeze(2)
    tx = env.p_x.unsqueeze(1); ty = env.p_y.unsqueeze(1)
    d = torch.sqrt((sx - tx) ** 2 + (sy - ty) ** 2)
    dmask = torch.where(valid_tgt.unsqueeze(1), d, torch.full_like(d, BIG))
    bestd, bestidx = dmask.min(2)
    has_tgt = bestd < BIG * 0.5
    btx = env.p_x.gather(1, bestidx); bty = env.p_y.gather(1, bestidx)
    angle = torch.atan2(bty - env.p_y, btx - env.p_x)
    go = base & has_tgt
    ships = torch.where(go, half, torch.zeros_like(half))
    commit = go.to(DTYPE)
    return angle, ships, commit

## 9. Env -- `step` (one tick; mirrors `gpu_env.cpp::step`)

Order: decode ego action (sequential per-planet ship budget) -> opponent launches (scripted
or self-play neural) -> deduct ships -> place fleets -> production -> planet orbit -> fleet
move + swept collision -> OOB/sun removal -> two-player combat -> increment step. **Comet
blocks omitted.** Returns the per-env reward ingredients (`StepOut`).

In [ ]:
class StepOut: pass

def env_step(env, ego_action, opponent, act_threshold, opp_action=None):
    '''One tick. ego_action (B,Ec,3K). opp_action (B,Ec,3K) for self-play, else scripted.'''
    B, Ec, Fc = env.B, env.Ec, env.Fc
    K = ego_action.shape[2] // 3
    vmax = env.vmax
    dev = env.dev
    ego, enemy = 0, 1

    prod_ego0 = (env.p_prod * (env.p_owner == float(ego)).to(DTYPE) * (env.p_alive > 0.5).to(DTYPE)).sum(1)
    prod_enemy0 = (env.p_prod * (env.p_owner == float(enemy)).to(DTYPE) * (env.p_alive > 0.5).to(DTYPE)).sum(1)
    ego_owned0 = (env.p_owner == float(ego)) & (env.p_alive > 0.5)

    # --- decode ego action (sequential per-planet budget) ---
    legal = (env.p_owner == float(ego)) & (env.p_alive > 0.5) & (env.p_ships > 0.0)
    S = env.p_ships
    remaining = S.clone()
    e_ang, e_shp, e_can = [], [], []
    invalid = torch.zeros(B, dtype=DTYPE, device=dev)
    launches = torch.zeros(B, dtype=DTYPE, device=dev)
    for k in range(K):
        dx = ego_action[:, :, 3 * k] * 2.0 - 1.0
        dy = ego_action[:, :, 3 * k + 1] * 2.0 - 1.0
        phi = ego_action[:, :, 3 * k + 2]
        commit = phi >= act_threshold
        n = torch.floor(phi * S)
        ok = commit & legal & (n >= 1.0) & (remaining >= n)
        inv_k = commit & (~ok)
        remaining = remaining - torch.where(ok, n, torch.zeros_like(n))
        invalid = invalid + inv_k.to(DTYPE).sum(1)
        launches = launches + ok.to(DTYPE).sum(1)
        e_ang.append(torch.atan2(dy, dx))
        e_shp.append(torch.where(ok, n, torch.zeros_like(n)))
        e_can.append(ok.to(DTYPE))

    # valid launches: committed ego launches whose heading lands on a planet (pre-step)
    valid = torch.zeros(B, dtype=DTYPE, device=dev)
    for k in range(K):
        can = e_can[k] > 0.5
        ox = env.p_x + torch.cos(e_ang[k]) * (env.p_radius + 0.1)
        oy = env.p_y + torch.sin(e_ang[k]) * (env.p_radius + 0.1)
        tgt, _ = fleet_target_batch(env, ox, oy, e_ang[k], e_shp[k].clamp_min(1.0), vmax)
        lands = (tgt >= 0) & can
        valid = valid + lands.to(DTYPE).sum(1)

    # --- opponent launches ---
    slot_idx = torch.arange(Ec, dtype=torch.long, device=dev).unsqueeze(0).expand(B, Ec)
    o_ded = torch.zeros(B, Ec, dtype=DTYPE, device=dev)
    if opp_action is None:
        o_ang, o_shp, o_can = opponent_action(env, opponent)
        o_ang_t, o_shp_t, o_can_t, o_slot_t = o_ang, o_shp, o_can, slot_idx.to(DTYPE)
        o_ded = o_shp
        o_slot_t = slot_idx
    else:  # self-play: decode player-1 (B,Ec,3K) exactly like the ego decode
        legal1 = (env.p_owner == float(enemy)) & (env.p_alive > 0.5) & (env.p_ships > 0.0)
        S1 = env.p_ships; rem1 = env.p_ships.clone()
        o_ang_k, o_shp_k, o_can_k = [], [], []
        for k in range(K):
            dx = opp_action[:, :, 3 * k] * 2.0 - 1.0
            dy = opp_action[:, :, 3 * k + 1] * 2.0 - 1.0
            phi = opp_action[:, :, 3 * k + 2]
            commit = phi >= act_threshold
            n = torch.floor(phi * S1)
            ok = commit & legal1 & (n >= 1.0) & (rem1 >= n)
            rem1 = rem1 - torch.where(ok, n, torch.zeros_like(n))
            o_ang_k.append(torch.atan2(dy, dx))
            o_shp_k.append(torch.where(ok, n, torch.zeros_like(n)))
            o_can_k.append(ok.to(DTYPE))
            o_ded = o_ded + o_shp_k[-1]
        o_ang_t = torch.stack(o_ang_k, 2).reshape(B, Ec * K)
        o_shp_t = torch.stack(o_shp_k, 2).reshape(B, Ec * K)
        o_can_t = torch.stack(o_can_k, 2).reshape(B, Ec * K)
        o_slot_t = slot_idx.unsqueeze(2).expand(B, Ec, K).reshape(B, Ec * K)

    # --- deduct ships (ego per-k + opponent) ---
    ded = torch.zeros(B, Ec, dtype=DTYPE, device=dev)
    for k in range(K):
        ded = ded + e_shp[k]
    ded = ded + o_ded
    env.p_ships = env.p_ships - ded

    # --- place fleets (ego planet-major, then opponent) ---
    e_ang_t = torch.stack(e_ang, 2).reshape(B, Ec * K)
    e_shp_t = torch.stack(e_shp, 2).reshape(B, Ec * K)
    e_can_t = torch.stack(e_can, 2).reshape(B, Ec * K)
    e_slot = slot_idx.unsqueeze(2).expand(B, Ec, K).reshape(B, Ec * K)
    Le = Ec * K; Lo = o_ang_t.shape[1]; Ltot = Le + Lo
    owner = torch.cat([torch.full((B, Le), float(ego), dtype=DTYPE, device=dev),
                       torch.full((B, Lo), float(enemy), dtype=DTYPE, device=dev)], 1)
    from_slot = torch.cat([e_slot, o_slot_t], 1)
    angle = torch.cat([e_ang_t, o_ang_t], 1)
    ships = torch.cat([e_shp_t, o_shp_t], 1)
    commit = torch.cat([e_can_t, o_can_t], 1)
    seq = (env.step_ct * float(Ltot + 1)).unsqueeze(1) + \
          torch.arange(Ltot, dtype=DTYPE, device=dev).unsqueeze(0)
    launch_fleets(env, owner, from_slot, angle, ships, commit, seq)

    # --- production ---
    env.p_ships = env.p_ships + env.p_prod * (env.p_owner != -1.0).to(DTYPE) * (env.p_alive > 0.5).to(DTYPE)

    # --- planet new positions (orbit) ---
    stepf = env.step_ct
    dxc = env.p_init_x - CENTER; dyc = env.p_init_y - CENTER
    r = torch.sqrt(dxc * dxc + dyc * dyc)
    ia = torch.atan2(dyc, dxc)
    ca = ia + env.ang_vel.unsqueeze(1) * stepf.unsqueeze(1)
    rot = env.p_rotates > 0.5
    nx = torch.where(rot, CENTER + r * torch.cos(ca), env.p_x)
    ny = torch.where(rot, CENTER + r * torch.sin(ca), env.p_y)
    old_px, old_py = env.p_x, env.p_y

    # --- fleet movement + swept collision against planet paths ---
    falive = env.f_alive > 0.5
    speed = fleet_speed_t(env.f_ships, vmax)
    fox, foy = env.f_x, env.f_y
    fnx = fox + torch.cos(env.f_angle) * speed
    fny = foy + torch.sin(env.f_angle) * speed
    Ax = fox.unsqueeze(2); Ay = foy.unsqueeze(2)
    Bx = fnx.unsqueeze(2); By = fny.unsqueeze(2)
    P0x = old_px.unsqueeze(1); P0y = old_py.unsqueeze(1)
    P1x = nx.unsqueeze(1); P1y = ny.unsqueeze(1)
    rad = env.p_radius.unsqueeze(1)
    palive = (env.p_alive.unsqueeze(1) > 0.5)
    d0x = Ax - P0x; d0y = Ay - P0y
    dvx = (Bx - Ax) - (P1x - P0x); dvy = (By - Ay) - (P1y - P0y)
    a = dvx * dvx + dvy * dvy
    b = 2.0 * (d0x * dvx + d0y * dvy)
    c = d0x * d0x + d0y * d0y - rad * rad
    disc = b * b - 4.0 * a * c
    sq = torch.sqrt(disc.clamp_min(0.0))
    t1 = (-b - sq) / (2.0 * a)
    t2 = (-b + sq) / (2.0 * a)
    hit_quad = (disc >= 0.0) & (t2 >= 0.0) & (t1 <= 1.0)
    hit_lin = (a < 1e-12) & (c <= 0.0)
    hit = torch.where(a < 1e-12, hit_lin, hit_quad)
    hit = hit & palive & falive.unsqueeze(2)        # (no comet check: comets omitted)
    slotf = torch.arange(Ec, dtype=DTYPE, device=dev).view(1, 1, Ec)
    order = torch.where(hit, slotf, torch.full_like(slotf, float(Ec)))
    fh_v, tgt_slot = order.min(2)
    has_hit = fh_v < float(Ec)

    # OOB / sun removal (point-to-segment to sun center)
    oob = (fnx < 0.0) | (fnx > BOARD_SIZE) | (fny < 0.0) | (fny > BOARD_SIZE)
    vx, vy, wx, wy = fox, foy, fnx, fny
    l2 = (vx - wx) ** 2 + (vy - wy) ** 2
    tt = ((CENTER - vx) * (wx - vx) + (CENTER - vy) * (wy - vy)) / l2.clamp_min(1e-12)
    tt = tt.clamp(0.0, 1.0)
    prx = vx + tt * (wx - vx); pry = vy + tt * (wy - vy)
    sundist = torch.sqrt((CENTER - prx) ** 2 + (CENTER - pry) ** 2)
    sun_hit = (l2 > 0.0) & (sundist < SUN_RADIUS)
    sun_pt = torch.sqrt((CENTER - vx) ** 2 + (CENTER - vy) ** 2) < SUN_RADIUS
    sun_hit = torch.where(l2 > 0.0, sun_hit, sun_pt)

    remove_fleet = falive & (has_hit | oob | sun_hit)
    contributes = falive & has_hit
    # EGO fleets that hit a planet this tick (reward: base + ships)
    ego_hit = contributes & (env.f_owner == float(ego))
    fleet_hits = ego_hit.to(DTYPE).sum(1)
    fleet_hit_ships = (env.f_ships * ego_hit.to(DTYPE)).sum(1)

    # --- combat: scatter arriving ships per owner, then elementwise resolve ---
    arr0 = torch.zeros(B, Ec, dtype=DTYPE, device=dev)
    arr1 = torch.zeros(B, Ec, dtype=DTYPE, device=dev)
    cf = contributes.to(DTYPE)
    s0 = env.f_ships * cf * (env.f_owner == float(ego)).to(DTYPE)
    s1 = env.f_ships * cf * (env.f_owner == float(enemy)).to(DTYPE)
    tslot = tgt_slot.clamp(0, Ec - 1)
    arr0.scatter_add_(1, tslot, s0)
    arr1.scatter_add_(1, tslot, s1)
    has0 = arr0 > 0.0; has1 = arr1 > 0.0
    top = torch.maximum(arr0, arr1); second = torch.minimum(arr0, arr1)
    both = has0 & has1
    surv_ships = torch.where(both, top - second, top)
    tie = both & (arr0 == arr1)
    surv_ships = torch.where(tie, torch.zeros_like(surv_ships), surv_ships)
    surv_owner = torch.full((B, Ec), -1.0, dtype=DTYPE, device=dev)
    surv_owner = torch.where(arr0 > arr1, torch.zeros_like(surv_owner), surv_owner)
    surv_owner = torch.where(arr1 > arr0, torch.ones_like(surv_owner), surv_owner)
    any_arr = has0 | has1
    apply = any_arr & (surv_ships > 0.0) & (env.p_alive > 0.5)
    same = (env.p_owner == surv_owner)
    reinforce = apply & same
    attack = apply & (~same)
    env.p_ships = torch.where(reinforce, env.p_ships + surv_ships, env.p_ships)
    after = env.p_ships - surv_ships
    flips = attack & (after < 0.0)
    env.p_ships = torch.where(attack, torch.where(after < 0.0, -after, after), env.p_ships)
    env.p_owner = torch.where(flips, surv_owner, env.p_owner)

    # --- apply planet positions; clear removed fleets ---
    env.p_x = nx; env.p_y = ny
    keep = falive & (~remove_fleet)
    keepf = keep.to(DTYPE)
    env.f_alive = keepf
    env.f_owner = env.f_owner * keepf
    env.f_x = fnx * keepf; env.f_y = fny * keepf
    env.f_angle = env.f_angle * keepf
    env.f_ships = env.f_ships * keepf

    env.step_ct = env.step_ct + 1.0

    prod_ego1 = (env.p_prod * (env.p_owner == float(ego)).to(DTYPE) * (env.p_alive > 0.5).to(DTYPE)).sum(1)
    prod_enemy1 = (env.p_prod * (env.p_owner == float(enemy)).to(DTYPE) * (env.p_alive > 0.5).to(DTYPE)).sum(1)
    ego_owned1 = (env.p_owner == float(ego)) & (env.p_alive > 0.5)

    out = StepOut()
    out.invalid = invalid
    out.valid = valid
    out.launches = launches
    out.dprod_ego = prod_ego1 - prod_ego0
    out.dprod_enemy = prod_enemy1 - prod_enemy0
    out.captured = (ego_owned1 & (~ego_owned0)).to(DTYPE).sum(1)
    out.lost = (ego_owned0 & (~ego_owned1)).to(DTYPE).sum(1)
    out.fleet_hits = fleet_hits
    out.fleet_hit_ships = fleet_hit_ships
    return out

## 10. Policy net (mirrors `model/policy_net.cpp`)

Per-planet trunk: `proj` -> masked single-head self-attention over planets (pre-LayerNorm) ->
optional GLU -> `N_RES_BLOCKS` pre-LayerNorm residual MLP blocks -> final LayerNorm. A separate
board-globals embedding `g' = relu(g_embed(globals))` broadcast to two heads: a nonlinear mean
head `mu_h2(relu(mu_h1([tok; g'])))` and a state-dependent logstd head. Value head: masked-mean
pool of trunk tokens + `g'` -> MLP -> scalar `V`. Calm init: small `mu_h2` weights, negative
`phi` bias.

In [ ]:
class PolicyNet(nn.Module):
    def __init__(self, F, G, hidden, d_g, K, n_res_blocks, use_glu, std_state_dependent,
                 logstd_min, logstd_max, init_mu_scale, init_phi_bias):
        super().__init__()
        self.F, self.G, self.h, self.d_g, self.K = F, G, hidden, d_g, K
        self.nap = 3 * K
        self.use_glu = use_glu
        self.std_state_dependent = std_state_dependent
        self.logstd_min = logstd_min
        self.logstd_max = logstd_max  # mutable: the anneal mutates this each iter
        h = hidden
        self.proj = nn.Linear(F, h)
        self.attn_q = nn.Linear(h, h); self.attn_k = nn.Linear(h, h)
        self.attn_v = nn.Linear(h, h); self.attn_o = nn.Linear(h, h)
        self.ln_attn = nn.LayerNorm(h); self.ln_out = nn.LayerNorm(h)
        if use_glu:
            self.glu_gate = nn.Linear(h, h); self.glu_val = nn.Linear(h, h); self.glu_out = nn.Linear(h, h)
        self.res_a = nn.ModuleList([nn.Linear(h, h) for _ in range(n_res_blocks)])
        self.res_b = nn.ModuleList([nn.Linear(h, h) for _ in range(n_res_blocks)])
        self.ln_res = nn.ModuleList([nn.LayerNorm(h) for _ in range(n_res_blocks)])
        self.g_embed = nn.Linear(G, d_g)
        self.mu_h1 = nn.Linear(h + d_g, h)
        self.mu_h2 = nn.Linear(h, self.nap)
        if std_state_dependent:
            self.logstd_head = nn.Linear(h + d_g, self.nap)
        else:
            self.logstd_param = nn.Parameter(torch.zeros(self.nap))
        # value head (PPO critic)
        self.val_h1 = nn.Linear(h + d_g, h)
        self.val_h2 = nn.Linear(h, 1)
        # calm init: small final mu weights so outputs start ~= bias; negative phi bias
        with torch.no_grad():
            self.mu_h2.weight.mul_(init_mu_scale)
            self.mu_h2.bias.zero_()
            for k in range(K):
                self.mu_h2.bias[3 * k + 2].fill_(init_phi_bias)

    def forward(self, entities, entity_mask, action_mask, globals_):
        tok = self.proj(entities)                       # (B,E,d)
        # masked single-head self-attention over planets (pre-norm)
        xn = self.ln_attn(tok)
        Q = self.attn_q(xn); Kt = self.attn_k(xn); Vt = self.attn_v(xn)
        scores = torch.matmul(Q, Kt.transpose(1, 2)) / math.sqrt(self.h)   # (B,E,E)
        dead_key = (entity_mask < 0.5).unsqueeze(1)     # (B,1,E)
        scores = scores.masked_fill(dead_key, -1e9)
        attn = torch.matmul(torch.softmax(scores, -1), Vt)
        tok = tok + self.attn_o(attn)
        if self.use_glu:
            tok = tok + self.glu_out(self.glu_val(tok) * torch.sigmoid(self.glu_gate(tok)))
        for i in range(len(self.res_a)):
            tok = tok + self.res_b[i](torch.relu(self.res_a[i](self.ln_res[i](tok))))
        tok = self.ln_out(tok)
        B, E = tok.shape[0], tok.shape[1]
        gp = torch.relu(self.g_embed(globals_))         # (B,d_g)
        gpb = gp.unsqueeze(1).expand(B, E, self.d_g)
        hh = torch.cat([tok, gpb], -1)                  # (B,E,d+d_g)
        mean = self.mu_h2(torch.relu(self.mu_h1(hh)))   # (B,E,3K)
        if self.std_state_dependent:
            logstd = self.logstd_head(hh)
        else:
            logstd = self.logstd_param.view(1, 1, self.nap).expand(B, E, self.nap)
        logstd = logstd.clamp(self.logstd_min, self.logstd_max)
        # value: masked-mean pool of trunk tokens over live planets + g'
        em = entity_mask.unsqueeze(-1)
        pooled = (tok * em).sum(1) / em.sum(1).clamp_min(1.0)
        vh = torch.cat([pooled, gp], -1)
        value = self.val_h2(torch.relu(self.val_h1(vh))).squeeze(-1)   # (B,)
        return mean, logstd, value


def build_policy():
    return PolicyNet(F_DIM, G_DIM, HIDDEN, D_G, K_FLEETS, N_RES_BLOCKS, USE_GLU, STD_STATE_DEP,
                     LOGSTD_MIN, LOGSTD_MAX, INIT_MU_SCALE, INIT_PHI_BIAS).to(DEVICE)


@torch.no_grad()
def act(net, ent, em, am, gl, greedy=False):
    mean, logstd, value = net(ent, em, am, gl)
    dist = SquashedGaussian(mean, logstd)
    action = dist.greedy() if greedy else dist.sample()
    return action, dist.log_prob(action), value


def evaluate(net, ent, em, am, gl, action):
    mean, logstd, value = net(ent, em, am, gl)
    dist = SquashedGaussian(mean, logstd)
    return dist.log_prob(action), dist.entropy(), value

## 11. Rollout + reward assembly + GAE (mirrors `rollout.cpp::collect_gpu`, PPO branch)

Per-step reward channels (capture, first-N dispatch, fleet-hit capped), the outcome scattered
onto each env's last active step, then GAE: `A_t = delta + gamma*lambda*notdone*A_{t+1}`,
`delta = r + gamma*V'*notdone - V`. Advantages normalized over kept transitions; value targets
`= adv + V`. Reward + outcome scaled by `1/PPO_REWARD_SCALE` before GAE; logged returns are in
real units.

Opponent is uniform across the batch this iteration. `dead` = a side has no planets AND no
fleets (true terminal -> `notdone=0`); time cap -> `notdone=1` (bootstrap `V`).

In [ ]:
def settle(env):
    '''{s0, s1, side0_alive, side1_alive}: ships (planets+fleets) per side + alive flags.'''
    pa = env.p_alive > 0.5; fa = env.f_alive > 0.5
    o0 = (env.p_owner == 0.0) & pa; o1 = (env.p_owner == 1.0) & pa
    g0 = (env.f_owner == 0.0) & fa; g1 = (env.f_owner == 1.0) & fa
    s0 = (env.p_ships * o0.to(DTYPE)).sum(1) + (env.f_ships * g0.to(DTYPE)).sum(1)
    s1 = (env.p_ships * o1.to(DTYPE)).sum(1) + (env.f_ships * g1.to(DTYPE)).sum(1)
    return s0, s1, (o0.any(1) | g0.any(1)), (o1.any(1) | g1.any(1))


def collect_ppo(env, net, world_pool, cursor, stage, opp_snapshot, rng):
    '''Collect one PPO iteration on `B` envs. Returns a trajectory dict + stats + cursor.'''
    Bn, Ec, T = B, env.Ec, env.T   # Bn from the config (env.B is only set after reset, below)
    nap = 3 * K_FLEETS
    dev = env.dev

    # opponent: 4-stage curriculum (noop / random / starter / self-play+starter)
    selfplay = False
    if stage == 1:
        opp = 2
    elif stage == 2:
        opp = 0
    elif stage == 3:
        opp = 1
    else:  # stage 4
        opp = 1
        selfplay = (opp_snapshot is not None) and (rng.random() < SELFPLAY_PROB)

    # PPO: maximize world diversity (no group sharing)
    worlds = [world_pool[(cursor + i) % len(world_pool)] for i in range(Bn)]
    cursor += Bn
    env.reset(worlds)

    # host buffers (CPU)
    ent_buf = torch.zeros(T, Bn, Ec, F_DIM)
    em_buf = torch.zeros(T, Bn, Ec); am_buf = torch.zeros(T, Bn, Ec)
    gl_buf = torch.zeros(T, Bn, G_DIM); act_buf = torch.zeros(T, Bn, Ec, nap)
    oldlp_buf = torch.zeros(T, Bn); valid_buf = torch.zeros(T, Bn)
    rew_buf = torch.zeros(T, Bn); val_buf = torch.zeros(T, Bn); done_buf = torch.zeros(T, Bn)

    active = torch.ones(Bn, device=dev)
    outcome = torch.zeros(Bn, device=dev)
    inv_sum = torch.zeros(Bn, device=dev); lnch_sum = torch.zeros(Bn, device=dev)
    valid_sum = torch.zeros(Bn, device=dev); step_sum = torch.zeros(Bn, device=dev)
    disp_acc = torch.zeros(Bn, device=dev); hit_acc = torch.zeros(Bn, device=dev)

    last_t = 0
    for t in range(T):
        last_t = t
        ent, em, am, gl = env_encode(env, 0)
        a_t, logp, value = act(net, ent, em, am, gl, greedy=False)
        ent_buf[t].copy_(ent, non_blocking=True); em_buf[t].copy_(em, non_blocking=True)
        am_buf[t].copy_(am, non_blocking=True); gl_buf[t].copy_(gl, non_blocking=True)
        act_buf[t].copy_(a_t, non_blocking=True); oldlp_buf[t].copy_(logp, non_blocking=True)
        valid_buf[t].copy_(active, non_blocking=True); val_buf[t].copy_(value, non_blocking=True)

        opp_act = None
        if selfplay:
            with torch.no_grad():
                o1e, o1m, o1a, o1g = env_encode(env, 1)
                opp_act, _, _ = act(net if opp_snapshot is None else opp_snapshot,
                                    o1e, o1m, o1a, o1g, greedy=False)
        out = env_step(env, a_t, opp, ACT_THRESHOLD, opp_act)
        a = active

        # --- event reward channels (docs/set-ups/1.md), a-masked ---
        cap_t = a * (CAPTURE_REWARD * (out.captured - out.lost))           # +/- per planet
        disp_room = (DISPATCH_COUNT - disp_acc).clamp_min(0.0)
        disp_t = a * (DISPATCH_REWARD * torch.minimum(out.launches, disp_room))
        disp_acc = disp_acc + a * out.launches
        hit_val = FLEET_HIT_BASE * out.fleet_hits + FLEET_HIT_SHIPW * out.fleet_hit_ships
        hit_room = (FLEET_HIT_CAP - hit_acc).clamp_min(0.0)
        hit_now = a * torch.minimum(hit_val, hit_room)
        hit_acc = hit_acc + a * hit_val

        r_t = cap_t + disp_t + hit_now                                    # spec reward only
        rew_buf[t].copy_(r_t / PPO_REWARD_SCALE, non_blocking=True)       # PPO value-target rescale

        inv_sum = inv_sum + a * out.invalid
        lnch_sum = lnch_sum + a * out.launches
        valid_sum = valid_sum + a * out.valid
        step_sum = step_sum + a

        s0, s1, side0, side1 = settle(env)
        step_now = env.step_ct
        dead = ~(side0 & side1)                       # true terminal: a side wiped out
        term = (step_now >= float(T - 2)) | dead
        done_buf[t].copy_(((active > 0.5) & dead).to(DTYPE), non_blocking=True)
        newly = (active > 0.5) & term
        outcome = torch.where(newly, torch.sign(s0 - s1), outcome)
        active = torch.where(term, torch.zeros_like(active), active)
        if (t & 15) == 15 and active.sum().item() == 0.0:
            break

    # settle any env still alive at the step cap
    s0, s1, side0, side1 = settle(env)
    outcome = torch.where(active > 0.5, torch.sign(s0 - s1), outcome)

    # bootstrap V(s_T) for envs alive at the cap (dead envs: notdone=0, unused)
    with torch.no_grad():
        fe, fm, fa_, fg = env_encode(env, 0)
        _, _, vT = act(net, fe, fm, fa_, fg, greedy=False)
        bootstrap = (vT * active).cpu()

    if dev.type == "cuda":
        torch.cuda.synchronize()
    Tu = last_t + 1
    keep = valid_buf[:Tu].reshape(-1).nonzero().squeeze(-1)

    outc = outcome.cpu()
    rew = rew_buf[:Tu]; val = val_buf[:Tu]; done = done_buf[:Tu]; alive = valid_buf[:Tu]
    # outcome: WIN decays by episode length, loss = -LOSS_PENALTY, draw = 0; scaled like r_t
    length = alive.sum(0)                                       # (B,) episode length
    win_val = torch.pow(torch.full_like(outc, WIN_DECAY), length) * WIN_BONUS
    Ot = torch.where(outc > 0.0, win_val,
                     torch.where(outc < 0.0, torch.full_like(outc, -LOSS_PENALTY),
                                 torch.zeros_like(outc)))
    Ot = Ot / PPO_REWARD_SCALE
    last_idx = (length - 1.0).clamp_min(0.0).long().unsqueeze(0)   # (1,B)
    rew.scatter_add_(0, last_idx, Ot.unsqueeze(0))
    # GAE backward
    adv = torch.zeros(Tu, Bn)
    A = torch.zeros(Bn); nextval = bootstrap.clone()
    for t in range(Tu - 1, -1, -1):
        al = alive[t]; notdone = 1.0 - done[t]
        delta = rew[t] + GAMMA * nextval * notdone - val[t]
        A = delta + GAMMA * GAE_LAMBDA * notdone * A
        adv[t] = A * al
        nextval = val[t]
        A = A * al
    ret_full = (adv + val).reshape(-1)
    adv_full = adv.reshape(-1)
    mean_return_log = (rew.sum(0).mean().item()) * PPO_REWARD_SCALE   # real units

    def sel(x):
        return x.index_select(0, keep)
    tb = {}
    tb["entities"] = sel(ent_buf[:Tu].reshape(Tu * Bn, Ec, F_DIM))
    tb["entity_mask"] = sel(em_buf[:Tu].reshape(Tu * Bn, Ec))
    tb["action_mask"] = sel(am_buf[:Tu].reshape(Tu * Bn, Ec))
    tb["globals"] = sel(gl_buf[:Tu].reshape(Tu * Bn, G_DIM))
    tb["action"] = sel(act_buf[:Tu].reshape(Tu * Bn, Ec, nap))
    tb["old_logp"] = sel(oldlp_buf[:Tu].reshape(Tu * Bn))
    adv_s = sel(adv_full)
    tb["advantage"] = (adv_s - adv_s.mean()) / (adv_s.std() + 1e-8)    # normalize over kept transitions
    tb["returns"] = sel(ret_full)
    tb["n"] = keep.shape[0]

    step_total = step_sum.sum().item()
    stats = {
        "mean_return": mean_return_log,
        "mean_len": step_total / Bn,
        "win_rate": (outcome > 0.0).to(DTYPE).mean().item(),
        "inv_per_step": (inv_sum.sum().item() / step_total) if step_total > 0 else 0.0,
        "lnch_per_step": (lnch_sum.sum().item() / step_total) if step_total > 0 else 0.0,
        "valid_per_step": (valid_sum.sum().item() / step_total) if step_total > 0 else 0.0,
        "transitions": tb["n"],
    }
    return tb, stats, cursor

## 12. PPO update (mirrors `grpo_trainer.cpp::update`)

Clipped surrogate + `VF_COEF*MSE(value, returns) - ENT_COEF*entropy`; grad clip; skip a
non-finite step. Entropy is **per-component** (`/ (E*3K)`) so `ent_coef` behaves like standard
PPO. Reports approx-KL, clipfrac, sigma.

In [ ]:
_HALF_LOG2PIE_C = 1.4189385332046727

def policy_surrogate(logp, oldlp, adv, clip):
    ratio = torch.exp(logp - oldlp)
    unclipped = ratio * adv
    clipped = torch.clamp(ratio, 1.0 - clip, 1.0 + clip) * adv
    return -torch.minimum(unclipped, clipped).mean()


def ppo_update(net, opt, tb):
    N = tb["n"]
    mb = MINIBATCHES
    mbsize = N // mb
    s = {"total": 0.0, "policy": 0.0, "vf": 0.0, "entropy": 0.0, "sigma": 0.0,
         "approx_kl": 0.0, "clipfrac": 0.0}
    if mbsize == 0:
        return s
    E = tb["entities"].shape[1]
    nap = 3 * K_FLEETS
    nsteps = 0
    for _ in range(UPDATE_EPOCHS):
        perm = torch.randperm(N)
        for b in range(mb):
            mi = perm[b * mbsize:(b + 1) * mbsize]
            ent = tb["entities"].index_select(0, mi).to(DEVICE)
            em = tb["entity_mask"].index_select(0, mi).to(DEVICE)
            am = tb["action_mask"].index_select(0, mi).to(DEVICE)
            gl = tb["globals"].index_select(0, mi).to(DEVICE)
            act_mb = tb["action"].index_select(0, mi).to(DEVICE)
            oldlp = tb["old_logp"].index_select(0, mi).to(DEVICE)
            adv = tb["advantage"].index_select(0, mi).to(DEVICE)
            ret = tb["returns"].index_select(0, mi).to(DEVICE)

            logp, entropy, value = evaluate(net, ent, em, am, gl, act_mb)
            pol = policy_surrogate(logp, oldlp, adv, CLIP)
            vloss = F.mse_loss(value, ret)
            ent_b = entropy.mean() / float(E * nap)         # per-component entropy
            loss = pol + VF_COEF * vloss - ENT_COEF * ent_b

            opt.zero_grad()
            loss.backward()
            gnorm = torch.nn.utils.clip_grad_norm_(net.parameters(), MAX_GRAD_NORM)
            if torch.isfinite(gnorm):
                opt.step()

            with torch.no_grad():
                ratio = torch.exp(logp - oldlp)
                s["total"] += loss.item(); s["policy"] += pol.item(); s["vf"] += vloss.item()
                s["entropy"] += ent_b.item()
                mean_logstd = ent_b.item() - _HALF_LOG2PIE_C
                s["sigma"] += math.exp(mean_logstd)
                s["approx_kl"] += (oldlp - logp).mean().item()
                s["clipfrac"] += (torch.abs(ratio - 1.0) > CLIP).to(DTYPE).mean().item()
            nsteps += 1
    if nsteps:
        for k in s:
            s[k] /= nsteps
    return s

## 13. The 4-stage train loop (mirrors `grpo_trainer.cpp::train`)

Iteration-driven curriculum, logstd-cap anneal (phase 1: force cap `LOGSTD_MAX -> LOGSTD_MAX_END`
over `SIGMA_DECAY_ITERS`; phase 2: release to `LOGSTD_MAX_POST`), stage-4 self-play snapshot
(refreshed every `SELFPLAY_REFRESH` iters). Per-iter heartbeat: stage, real return, launches/step,
approx-KL, clipfrac, sigma. Returns logged history for plotting.

In [ ]:
import copy

def stage_for_iter(it):
    s1, s2, s3 = STAGE1_ITERS, STAGE2_ITERS, STAGE3_ITERS
    if it < s1:
        return 1
    if it < s1 + s2:
        return 2
    if it < s1 + s2 + s3:
        return 3
    return 4


def anneal_logstd_max(net, it):
    Nd = SIGMA_DECAY_ITERS
    if Nd > 0:
        if it < Nd:
            f = it / Nd
            cur = LOGSTD_MAX + (LOGSTD_MAX_END - LOGSTD_MAX) * f
        else:
            cur = LOGSTD_MAX_POST
    else:
        cur = LOGSTD_MAX
    cur = max(cur, LOGSTD_MIN)
    net.logstd_max = cur
    return cur


def train(total_iters=TOTAL_ITERS, log_every=1):
    net = build_policy()
    opt = torch.optim.Adam(net.parameters(), lr=LR, eps=ADAM_EPS)
    env = GpuEnv(PLANET_CAP, FLEET_CAP, EPISODE_STEPS, SHIP_SPEED, DEVICE)
    pool = make_world_pool(N_WORLDS, base_seed=SEED)
    rng = random.Random(SEED * 2654435761 + 12345)
    cursor = 0
    opp_snapshot = None
    opp_set = False
    hist = {"iter": [], "stage": [], "return": [], "win_rate": [],
            "lnch_per_step": [], "approx_kl": [], "clipfrac": [], "sigma": []}

    for it in range(total_iters):
        t0 = time.time()
        sigma_cap = anneal_logstd_max(net, it)
        stage = stage_for_iter(it)
        # stage-4 self-play: (re)build frozen snapshot at entry & every SELFPLAY_REFRESH iters
        if stage == 4:
            if (not opp_set) or (it % max(1, SELFPLAY_REFRESH) == 0):
                opp_snapshot = copy.deepcopy(net).eval()
                for p in opp_snapshot.parameters():
                    p.requires_grad_(False)
                opp_set = True
        tb, rs, cursor = collect_ppo(env, net, pool, cursor, stage, opp_snapshot, rng)
        us = ppo_update(net, opt, tb)
        dt = time.time() - t0
        sps = rs["transitions"] / max(dt, 1e-9)

        hist["iter"].append(it + 1); hist["stage"].append(stage)
        hist["return"].append(rs["mean_return"]); hist["win_rate"].append(rs["win_rate"])
        hist["lnch_per_step"].append(rs["lnch_per_step"]); hist["approx_kl"].append(us["approx_kl"])
        hist["clipfrac"].append(us["clipfrac"]); hist["sigma"].append(us["sigma"])

        if (it + 1) % log_every == 0 or it == 0 or it == total_iters - 1:
            print("it%4d s%d | ret %8.2f wr %.2f | lnch/st %.2f valid/st %.2f inv/st %.2f | "
                  "kl %.3f cf %.2f sig %.3f | loss %7.3f (pol %.3f vf %.3f) | sps %5.0f"
                  % (it + 1, stage, rs["mean_return"], rs["win_rate"], rs["lnch_per_step"],
                     rs["valid_per_step"], rs["inv_per_step"], us["approx_kl"], us["clipfrac"],
                     us["sigma"], us["total"], us["policy"], us["vf"], sps))
    return net, hist

## 14. Run training

This is the heavy cell. With `QUICK_SMOKE=True` it is a short sanity run; flip `QUICK_SMOKE`
off (section 2) and re-run for the full spec. On a Colab/Kaggle T4 the full run takes hours --
reduce `B`, `HIDDEN`, `N_RES_BLOCKS`, or `TOTAL_ITERS` to fit your budget.

In [ ]:
net, hist = train(total_iters=TOTAL_ITERS, log_every=1)

## 15. Plot the logged curves (return / win-rate)

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 7))
it = hist["iter"]
ax[0, 0].plot(it, hist["return"]);       ax[0, 0].set_title("episode return (real units)")
ax[0, 0].set_xlabel("iter"); ax[0, 0].grid(alpha=0.3)
ax[0, 1].plot(it, hist["win_rate"]);     ax[0, 1].set_title("win rate (ego)")
ax[0, 1].set_xlabel("iter"); ax[0, 1].set_ylim(-0.02, 1.02); ax[0, 1].grid(alpha=0.3)
ax[1, 0].plot(it, hist["lnch_per_step"]); ax[1, 0].set_title("launches / step")
ax[1, 0].set_xlabel("iter"); ax[1, 0].grid(alpha=0.3)
ax[1, 1].plot(it, hist["sigma"], label="sigma")
ax[1, 1].plot(it, hist["clipfrac"], label="clipfrac")
ax[1, 1].plot(it, hist["approx_kl"], label="approx_kl")
ax[1, 1].set_title("exploration / update health"); ax[1, 1].set_xlabel("iter")
ax[1, 1].legend(); ax[1, 1].grid(alpha=0.3)
# stage boundaries
for a in ax.flat:
    for bnd in (STAGE1_ITERS, STAGE1_ITERS + STAGE2_ITERS, STAGE1_ITERS + STAGE2_ITERS + STAGE3_ITERS):
        if bnd <= max(it):
            a.axvline(bnd, color="k", ls=":", alpha=0.3)
plt.tight_layout(); plt.show()

## 16. Optional shape smoke test (default OFF)

A tiny `B=4` dry run of env + policy that checks tensor shapes and runs a couple of steps.
Guarded by `RUN_SMOKE=False` so it does not run heavy work on import / "Run all".

In [ ]:
RUN_SMOKE = False  # set True to run the shape sanity check

if RUN_SMOKE:
    senv = GpuEnv(PLANET_CAP, FLEET_CAP, 8, SHIP_SPEED, DEVICE)
    sworlds = make_world_pool(4, base_seed=123)
    senv.reset(sworlds)
    snet = build_policy()
    ent, em, am, gl = env_encode(senv, 0)
    assert ent.shape == (4, PLANET_CAP, F_DIM), ent.shape
    assert gl.shape == (4, G_DIM), gl.shape
    a_t, logp, value = act(snet, ent, em, am, gl)
    assert a_t.shape == (4, PLANET_CAP, 3 * K_FLEETS), a_t.shape
    assert logp.shape == (4,) and value.shape == (4,)
    print("encode/act OK | ent", tuple(ent.shape), "act", tuple(a_t.shape),
          "logp", tuple(logp.shape), "V", tuple(value.shape))
    for opp in (2, 0, 1):   # noop / random / starter
        out = env_step(senv, a_t, opp, ACT_THRESHOLD, None)
        print("step opp=%d | launches=%.0f valid=%.0f captured=%.0f hits=%.0f"
              % (opp, out.launches.sum(), out.valid.sum(), out.captured.sum(), out.fleet_hits.sum()))
    # self-play step
    o1e, o1m, o1a, o1g = env_encode(senv, 1)
    opp_a, _, _ = act(snet, o1e, o1m, o1a, o1g)
    out = env_step(senv, a_t, 1, ACT_THRESHOLD, opp_a)
    print("self-play step OK | step_ct", senv.step_ct[0].item())
    print("SMOKE OK")
else:
    print("smoke test disabled (set RUN_SMOKE=True to enable)")